This notebook is used to get the appropriate audio file names from Audioset's and FSD50k's training and evaluation sets that correspond to a given mid (Machine identifier) list. This list of filenames can then be used to construct appropriate dataset specific training and evaluation sets for further downstream machine learning. Corresponding label files can also be constructed.

In [17]:
import os
import zipfile as zf

import pandas as pd

import utils.utility_funcs as my_util

In [18]:


# Load in the respective metafiles for both audioset and fsd50k, keeping only relevant columns
PATH_TO_AS_TRAIN = r'C:\Users\mp431591\Documents\audioset_label_files\audioset_train_strong.tsv'
PATH_TO_AS_EVAL = r'C:\Users\mp431591\Documents\audioset_label_files\audioset_eval_strong.tsv'

PATH_TO_FSD_TRAIN = r'C:\Users\mp431591\Documents\FSD50k_label_files\dev.csv' # dev == train
PATH_TO_FSD_EVAL = r'C:\Users\mp431591\Documents\FSD50k_label_files\eval.csv'

audioset_train_meta = my_util.read_strong_audioset_metatable(PATH_TO_AS_TRAIN)
audioset_eval_meta = my_util.read_strong_audioset_metatable(PATH_TO_AS_EVAL)

fsd50k_train_meta = my_util.read_fsd50k_metatable(PATH_TO_FSD_TRAIN)
fsd50k_eval_meta = my_util.read_fsd50k_metatable(PATH_TO_FSD_EVAL)

audioset_eval_meta.head()

,segment_id,mids
0,s9d-2nhuJCQ_30000,/m/04rlf
1,s9d-2nhuJCQ_30000,/m/053hz1
2,s9d-2nhuJCQ_30000,/m/03qtwd
3,s9d-2nhuJCQ_30000,/m/01w250
4,s9d-2nhuJCQ_30000,/m/0l15bq


In [19]:
fsd50k_train_meta.head()

,fname,mids
0,64760,"/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf"
1,16399,"/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf"
2,16401,"/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf"
3,16402,"/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf"
4,16404,"/m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf"


In [20]:
# Explode the fsd50k table to make checking for valid rows easier
# Also conforms to the form of the Audioset data -> operations can be similar
# across datasets
fsd50k_train_meta_exploded = (fsd50k_train_meta.set_index(["fname"])
                              .apply(lambda x: x.str.split(',')
                                     .explode())
                                     .reset_index())
fsd50k_train_meta_exploded.head(10)

,fname,mids
0,64760,/m/02sgy
1,64760,/m/0342h
2,64760,/m/0fx80y
3,64760,/m/04szw
4,64760,/m/04rlf
5,16399,/m/02sgy
6,16399,/m/0342h
7,16399,/m/0fx80y
8,16399,/m/04szw
9,16399,/m/04rlf


In [21]:
fsd50k_eval_meta_exploded = (fsd50k_eval_meta.set_index(["fname"])
                              .apply(lambda x: x.str.split(',')
                                     .explode())
                                     .reset_index())
fsd50k_eval_meta_exploded.head(10)

,fname,mids
0,37199,/m/02sgy
1,37199,/m/0342h
2,37199,/m/0fx80y
3,37199,/m/04szw
4,37199,/m/04rlf
5,175151,/m/02sgy
6,175151,/m/0342h
7,175151,/m/0fx80y
8,175151,/m/04szw
9,175151,/m/04rlf


In [22]:
meta_tables = {"audioset_train_meta": audioset_train_meta,
               "audioset_eval_meta": audioset_eval_meta,
               "fsd50k_train_meta_exploded": fsd50k_train_meta_exploded,
               "fsd50k_eval_meta_exploded": fsd50k_eval_meta_exploded}


In [23]:
PATH_TO_MIDS_BY_AS_COUNT = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\mids_by_AStrain_counts.pckl'

# Load a mid to display name vocabulary as well
PATH_TO_AS_VOCAB = r'C:\Users\mp431591\Documents\audioset_label_files\mid_to_display_name.tsv'
audioset_mid_to_label_mapping = pd.read_csv(PATH_TO_AS_VOCAB,
                                     sep="\t", header=None, names=["mids", "display_name"])
# Make a copy to retain original table and refine the copy to a dict
as_mid_to_label_map_copy = audioset_mid_to_label_mapping.set_index('mids')
as_mid_to_display_dict = as_mid_to_label_map_copy.to_dict()['display_name']
as_mid_to_display_dict


{'/g/11b630rrvh': 'Kettle whistle',
 '/g/122z_qxw': 'Firecracker',
 '/m/01280g': 'Wild animals',
 '/m/012f08': 'Motor vehicle (road)',
 '/m/012n7d': 'Ambulance (siren)',
 '/m/012ndj': 'Fire engine, fire truck (siren)',
 '/m/012xff': 'Toothbrush',
 '/m/0130jx': 'Sink (filling or washing)',
 '/m/014yck': 'Aircraft engine',
 '/m/014zdl': 'Explosion',
 '/m/0150b9': 'Change ringing (campanology)',
 '/m/015jpf': 'Dial tone',
 '/m/015lz1': 'Singing',
 '/m/015p6': 'Bird',
 '/m/0160x5': 'Digestive',
 '/m/0174k2': 'Washing machine',
 '/m/018p4k': 'Cart',
 '/m/018w8': 'Basketball bounce',
 '/m/0193bn': 'Sonic boom',
 '/m/0195fx': 'Subway, metro, underground',
 '/m/0199g': 'Bicycle, tricycle',
 '/m/019jd': 'Boat, Water vehicle',
 '/m/01b82r': 'Sawing',
 '/m/01b9nn': 'Reverberation',
 '/m/01b_21': 'Cough',
 '/m/01bjv': 'Bus',
 '/m/01c194': 'Mantra',
 '/m/01d380': 'Drill',
 '/m/01d3sd': 'Snoring',
 '/m/01dwxx': 'Gull, seagull',
 '/m/01g50p': 'Railroad car, train wagon',
 '/m/01g90h': 'Stomach rumble

In [27]:
top_50_mids = my_util.pickle_load("pickle_objects/top_mids/top_50_mids.pckl")
mid_nrs = [30, 35, 40, 45, 50]
write_out_filenames = False
write_out_labels = True

for mid_nr in mid_nrs:

    pckl_name = f"top_{mid_nr}_mids.pckl"
    how_many_mids = str(pckl_name.split("_")[1])
    print(how_many_mids)
    mids = my_util.pickle_load("pickle_objects/top_mids/" + pckl_name)
    print(f"Is mids contained in the top 50 mids: {mids.issubset(top_50_mids)}")

    for table_name in meta_tables:

        split_table_name = table_name.split('_')
        dataset = split_table_name[0]
        datasplit = split_table_name[1]
        meta_table = meta_tables[table_name]

        # Valid filename generation

        filenames = ""
        filenames_name = f"{dataset}_{datasplit}_filenames_top{mid_nr}.txt"
        if dataset == 'audioset':
            filenames = my_util.get_valid_audioset_filenames_for_mids(meta_table, mids)
        else:
            filenames = my_util.get_valid_fsd50k_filenames_for_mids(meta_table, mids)
        print(f"Number of valid files for {dataset} {datasplit}: {len(filenames)}")

        if write_out_filenames:
            my_util.filenames_to_txt(filenames,
                         f"test_filenames/{filenames_name}",
                         dataset=dataset)
        
        # Label generation
        # Sorting by mid count makes it easier to compare between models with a different number of classes
        mids_by_AStrain_counts = my_util.pickle_load(PATH_TO_MIDS_BY_AS_COUNT)
        sorted_mids = sorted(mids, key=lambda x: mids_by_AStrain_counts[x], reverse=True)

        mid_to_multihot_mapping = {}
        multihot_to_mid_mapping = {}
        for idx, mid in enumerate(sorted_mids):
            mid_to_multihot_mapping[mid] = idx
            multihot_to_mid_mapping[idx] = mid
            print(f"{mid} corresponds to multihot label position {idx}")

        # A lot of functionality is behind this, so take a look at utility_funcs.py if needed.
        labels = my_util.get_multihot_labels_per_file(meta_table,
                                                    mids, mid_to_multihot_mapping, dataset=dataset)
        labels_name = f"{dataset}_{datasplit}_labels_{mid_nr}_by_count.pckl"
        if write_out_labels:
            my_util.pickle_save(f"pickle_objects/labels_by_count/{labels_name}", labels)


30
Is mids contained in the top 50 mids: True
Number of valid files for audioset train: 91785
/m/05zppz corresponds to multihot label position 0
/m/04rlf corresponds to multihot label position 1
/t/dd00077 corresponds to multihot label position 2
/m/07qjznt corresponds to multihot label position 3
/m/0lyf6 corresponds to multihot label position 4
/m/02zsn corresponds to multihot label position 5
/m/03m9d0z corresponds to multihot label position 6
/m/09l8g corresponds to multihot label position 7
/m/01h8n0 corresponds to multihot label position 8
/m/01j3sz corresponds to multihot label position 9
/m/07qcpgn corresponds to multihot label position 10
/m/020bb7 corresponds to multihot label position 11
/m/0ytgt corresponds to multihot label position 12
/t/dd00003 corresponds to multihot label position 13
/m/07q2z82 corresponds to multihot label position 14
/m/07p6fty corresponds to multihot label position 15
/m/07pbtc8 corresponds to multihot label position 16
/m/03qtwd corresponds to mult

In [ ]:
# Cross-checking if found number of files based on top X filenames
# equals the number of files found through computed valid filenames list
# Essentially, makes it easy to check that the mid filtering works as intended
audioset_eval_meta_fnames = audioset_eval_meta.copy()
audioset_eval_meta_fnames = audioset_eval_meta_fnames.drop_duplicates()
audioset_eval_meta_fnames = audioset_eval_meta_fnames[audioset_eval_meta_fnames.mids.isin(mids)]
audioset_eval_meta_fnames = set(audioset_eval_meta_fnames['segment_id'])
audioset_eval_meta_fnames = {('_').join(seg.split('_')[0:-1]) for seg in audioset_eval_meta_fnames}
print(len(audioset_eval_meta_fnames))

found_files = 0

PATH_TO_AS_EVAL_AUDIO = r'C:\Users\mp431591\Documents\audioset_eval_files'
with os.scandir(PATH_TO_AS_EVAL_AUDIO) as blob_dir:
    # For each blob
    for idx, entry in enumerate(blob_dir):
        with zf.ZipFile(entry.path, 'r') as archive:
            archive_path = archive.filename

            for zip_info in archive.infolist():

                if zip_info.is_dir():
                    continue
                # Break archive directory structures
                zip_info.filename = os.path.basename(zip_info.filename) 
                filename = zip_info.filename
                clean_file_name = filename.rstrip(".wav")
                clean_file_name_wo_Y = clean_file_name[1::] # remove first character (Y)
                print(clean_file_name_wo_Y)

                if clean_file_name_wo_Y in audioset_eval_meta_fnames:
                    found_files += 1
print(found_files)


In [ ]:
# Here locally available files can be intersected with files with labels, since having the 
# whole dataset available locally is untenable
PATH_TO_LOCAL_AUDIOSET_00_FILES = r'C:\Users\mp431591\Documents\audioset_unbalanced_test\audios\unbalanced_train_segments\unbalanced_train_segments_part00'
test_audioset_files = os.scandir(PATH_TO_LOCAL_AUDIOSET_00_FILES)

for fname in test_audioset_files:
    cleaned_name = ('.').join(fname.name.split('.')[0:-1])[1::]
    if cleaned_name in audioset_train_labels.keys():
        print(cleaned_name)

In [14]:
test_mid_nr = 35
test_mids = my_util.pickle_load(f"pickle_objects/top_mids/top_{test_mid_nr}_mids.pckl")
sorted_mids = sorted(test_mids)

dataset = 'audioset'
datasplit = 'train'
test_file = ''

test_label_dict = my_util.pickle_load(f"C:/Users/mp431591/Documents/work_code/cl_30/continual_learning/pickle_objects/labels/{dataset}_{datasplit}_labels_{test_mid_nr}_dict.pckl")

mid_to_multihot_mapping = {}
multihot_to_mid_mapping = {}
for idx, mid in enumerate(sorted_mids):
    mid_to_multihot_mapping[mid] = idx
    multihot_to_mid_mapping[idx] = mid

#
test_label = [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]

test_translation = my_util.multihot_labels_translation(test_label, multihot_to_mid_mapping, as_mid_to_display_dict)
print(test_translation)

['Music', 'Female singing']
